# RAG v2: Hybrid Search + Re-ranking + Groq

**Goal:** Improve my basic RAG (embeddings only) built over a short text document (RAG notes plus a waste-management annotation overview).

**What I added:**
- Hybrid search: embeddings + BM25 keyword search, combined with Reciprocal Rank Fusion
- A re-ranker (cross-encoder) to order the top candidates
- Answer generation with Groq (openai/gpt-oss-20b), grounded in the retrieved chunks
- A FastAPI /ask endpoint (tested inside the notebook)

**Result (on my own 6-question test set):** hybrid search found the right chunk first more often than embeddings alone (4/6 to 5/6). The re-ranker did not add gains on this small set.



In [21]:
!pip install langchain langchain-community langchain-text-splitters chromadb sentence-transformers -q

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
!ls "/content/drive/MyDrive/rag_notes_qa"

data  notebooks  README.md


In [24]:
from langchain_text_splitters import CharacterTextSplitter

with open("/content/drive/MyDrive/rag_notes_qa/data/combined_notes.txt", "r") as f:
    text = f.read()

splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50, separator="\n\n")
chunks = splitter.split_text(text)

print(f"Total chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---\n{chunk}")

Total chunks: 9

--- Chunk 1 ---
RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, RAG retrieves relevant documents from an external knowledge source and uses them as context before generating an answer. This reduces hallucination and allows the model to answer using up-to-date or private data.

--- Chunk 2 ---
The RAG pipeline has three main steps. First, indexing: documents are split into smaller chunks, each chunk is converted into a numeric vector using an embedding model, and these vectors are stored in a vector database. Second, retrieval: when a user asks a question, the question is also converted into an embedding, and the vector database is searched for the most similar chunks using cosine similarity. Third, generation: the retrieved chunks are combined with the original question into a single prompt, which is passed to a language model to produ

In [25]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = Chroma.from_texts(chunks, embedding_model)

print("Vector database created successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database created successfully


In [26]:
query = "What is cosine similarity?"

results = vector_db.similarity_search(query, k=2)

for i, r in enumerate(results):
    print(f"\n--- Retrieved Chunk {i+1} ---\n{r.page_content}")


--- Retrieved Chunk 1 ---
Embeddings are numeric representations of text that capture meaning. Texts with similar meaning have embeddings that are close together in vector space, even if they use completely different words. This is measured using cosine similarity, which calculates the angle between two vectors. A smaller angle means higher similarity.

--- Retrieved Chunk 2 ---
Embeddings are numeric representations of text that capture meaning. Texts with similar meaning have embeddings that are close together in vector space, even if they use completely different words. This is measured using cosine similarity, which calculates the angle between two vectors. A smaller angle means higher similarity.


In [27]:
query2 = "How is waste sorted and annotated for AI models?"

results2 = vector_db.similarity_search(query2, k=2)

for i, r in enumerate(results2):
    print(f"\n--- Retrieved Chunk {i+1} ---\n{r.page_content}")


--- Retrieved Chunk 1 ---
High-quality annotated data is essential for waste sorting robots and AI-powered recycling systems to work accurately, reduce contamination rates, and improve overall recycling efficiency.

--- Retrieved Chunk 2 ---
High-quality annotated data is essential for waste sorting robots and AI-powered recycling systems to work accurately, reduce contamination rates, and improve overall recycling efficiency.


In [28]:
!pip install langchain langchain-community langchain-text-splitters chromadb sentence-transformers rank_bm25 -q

In [29]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

query = "What is cosine similarity?"
scores = bm25.get_scores(query.lower().split())

top = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:2]
for rank, i in enumerate(top, 1):
    print(f"\n--- Keyword result {rank} (chunk {i+1}, score {scores[i]:.2f}) ---\n{chunks[i]}")


--- Keyword result 1 (chunk 1, score 1.95) ---
RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, RAG retrieves relevant documents from an external knowledge source and uses them as context before generating an answer. This reduces hallucination and allows the model to answer using up-to-date or private data.

--- Keyword result 2 (chunk 3, score 1.48) ---
Embeddings are numeric representations of text that capture meaning. Texts with similar meaning have embeddings that are close together in vector space, even if they use completely different words. This is measured using cosine similarity, which calculates the angle between two vectors. A smaller angle means higher similarity.


In [30]:
import re
from rank_bm25 import BM25Okapi

stop = {"what","is","the","a","an","of","to","and","in","how","does","do","for","are","it","this","that","with","as","by","on"}
def tok(text):
    return [w for w in re.findall(r"[a-z0-9\-]+", text.lower()) if w not in stop]

bm25 = BM25Okapi([tok(c) for c in chunks])

query = "What is cosine similarity?"
scores = bm25.get_scores(tok(query))

top = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:2]
for rank, i in enumerate(top, 1):
    print(f"\n--- Keyword result {rank} (chunk {i+1}, score {scores[i]:.2f}) ---\n{chunks[i]}")


--- Keyword result 1 (chunk 3, score 1.97) ---
Embeddings are numeric representations of text that capture meaning. Texts with similar meaning have embeddings that are close together in vector space, even if they use completely different words. This is measured using cosine similarity, which calculates the angle between two vectors. A smaller angle means higher similarity.

--- Keyword result 2 (chunk 2, score 1.28) ---
The RAG pipeline has three main steps. First, indexing: documents are split into smaller chunks, each chunk is converted into a numeric vector using an embedding model, and these vectors are stored in a vector database. Second, retrieval: when a user asks a question, the question is also converted into an embedding, and the vector database is searched for the most similar chunks using cosine similarity. Third, generation: the retrieved chunks are combined with the original question into a single prompt, which is passed to a language model to produce the final answer.


In [31]:
def hybrid_search(query, k=2, rrf_k=60):
    # 1) semantic ranking (embeddings)
    sem_docs = vector_db.similarity_search(query, k=len(chunks))
    sem_rank = [chunks.index(d.page_content) for d in sem_docs]
    # 2) keyword ranking (BM25)
    kw_scores = bm25.get_scores(tok(query))
    kw_rank = sorted(range(len(chunks)), key=lambda i: kw_scores[i], reverse=True)
    # 3) fuse both rankings
    fused = {}
    for ranking in (sem_rank, kw_rank):
        for pos, i in enumerate(ranking):
            fused[i] = fused.get(i, 0) + 1 / (rrf_k + pos + 1)
    return sorted(fused, key=fused.get, reverse=True)[:k]

for q in ["What is cosine similarity?", "inter-annotator agreement checks"]:
    print(f"\n=== {q} ===")
    for i in hybrid_search(q):
        print(f"[chunk {i+1}] {chunks[i][:110]}...")


=== What is cosine similarity? ===
[chunk 3] Embeddings are numeric representations of text that capture meaning. Texts with similar meaning have embedding...
[chunk 4] Vector databases such as Chroma, Pinecone, and FAISS are optimized for fast similarity search across large num...

=== inter-annotator agreement checks ===
[chunk 8] Annotators must also handle edge cases such as overlapping items, partially visible objects, dirty or damaged ...
[chunk 7] Annotation for waste management datasets typically involves drawing bounding boxes or segmentation masks aroun...


In [32]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def hybrid_rerank(query, fetch=5, k=2):
    cand = hybrid_search(query, k=fetch)
    scores = reranker.predict([(query, chunks[i]) for i in cand])
    return sorted(zip(cand, scores), key=lambda x: x[1], reverse=True)[:k]

for q in ["What is cosine similarity?", "inter-annotator agreement checks"]:
    print(f"\n=== {q} ===")
    for i, s in hybrid_rerank(q):
        print(f"[chunk {i+1}] score {s:.2f} | {chunks[i][:100]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


=== What is cosine similarity? ===
[chunk 3] score 3.28 | Embeddings are numeric representations of text that capture meaning. Texts with similar meaning have...
[chunk 2] score 1.35 | The RAG pipeline has three main steps. First, indexing: documents are split into smaller chunks, eac...

=== inter-annotator agreement checks ===
[chunk 8] score 7.34 | Annotators must also handle edge cases such as overlapping items, partially visible objects, dirty o...
[chunk 7] score -9.77 | Annotation for waste management datasets typically involves drawing bounding boxes or segmentation m...


In [33]:
for n, c in enumerate(chunks, 1):
    print(f"\n[{n}] {c}")


[1] RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, RAG retrieves relevant documents from an external knowledge source and uses them as context before generating an answer. This reduces hallucination and allows the model to answer using up-to-date or private data.

[2] The RAG pipeline has three main steps. First, indexing: documents are split into smaller chunks, each chunk is converted into a numeric vector using an embedding model, and these vectors are stored in a vector database. Second, retrieval: when a user asks a question, the question is also converted into an embedding, and the vector database is searched for the most similar chunks using cosine similarity. Third, generation: the retrieved chunks are combined with the original question into a single prompt, which is passed to a language model to produce the final answer.

[3] Embeddings are

In [34]:
golden = [
    ("What is cosine similarity?", 3),
    ("inter-annotator agreement checks", 8),
    ("Which vector databases are fast for similarity search?", 4),
    ("why quality check is used in annotation",7),
    ("what is need of error free data?", 9),
    ("differnt step of rag pipeline hirarichy", 2)

]

def v1(q): return [chunks.index(d.page_content) for d in vector_db.similarity_search(q, k=2)]
def hy(q): return hybrid_search(q, k=2)
def rr(q): return [i for i, _ in hybrid_rerank(q, k=2)]

for name, fn in [("v1 embeddings only", v1), ("hybrid", hy), ("hybrid + rerank", rr)]:
    top1 = sum(fn(q)[0] == a - 1 for q, a in golden)
    top2 = sum((a - 1) in fn(q) for q, a in golden)
    print(f"{name:20s} top-1: {top1}/{len(golden)}   top-2: {top2}/{len(golden)}")

v1 embeddings only   top-1: 4/6   top-2: 4/6
hybrid               top-1: 4/6   top-2: 5/6
hybrid + rerank      top-1: 5/6   top-2: 5/6


In [35]:
for q, a in golden:
    print(f"\nQ: {q}  | correct: chunk {a}")
    print("  v1     :", [i+1 for i in v1(q)])
    print("  hybrid :", [i+1 for i in hy(q)])
    print("  rerank :", [i+1 for i in rr(q)])


Q: What is cosine similarity?  | correct: chunk 3
  v1     : [3, 3]
  hybrid : [3, 4]
  rerank : [3, 2]

Q: inter-annotator agreement checks  | correct: chunk 8
  v1     : [8, 8]
  hybrid : [8, 7]
  rerank : [8, 7]

Q: Which vector databases are fast for similarity search?  | correct: chunk 4
  v1     : [4, 4]
  hybrid : [4, 3]
  rerank : [4, 2]

Q: why quality check is used in annotation  | correct: chunk 7
  v1     : [8, 8]
  hybrid : [8, 7]
  rerank : [8, 9]

Q: what is need of error free data?  | correct: chunk 9
  v1     : [7, 7]
  hybrid : [8, 7]
  rerank : [9, 7]

Q: differnt step of rag pipeline hirarichy  | correct: chunk 2
  v1     : [2, 2]
  hybrid : [2, 5]
  rerank : [2, 5]


In [36]:
q = "why quality check is used in annotation"
cand = hybrid_search(q, k=5)
for i, s in sorted(zip(cand, reranker.predict([(q, chunks[i]) for i in cand])), key=lambda x: -x[1]):
    print(f"chunk {i+1}: {s:.2f}")

chunk 8: 4.53
chunk 9: 4.36
chunk 7: 4.35
chunk 5: -9.03
chunk 1: -10.79


In [37]:
!pip install groq -q

from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
#MODEL = "llama-3.3-70b-versatile"
# Connection test (kept for reference)
#r = client.chat.completions.create(
  #  model=MODEL,
   # messages=[{"role": "user", "content": "Say hello in one short line."}],
#)
#print(r.choices[0].message.content)

In [38]:
for m in client.models.list().data:
    print(m.id)

whisper-large-v3
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-v1-english
openai/gpt-oss-safeguard-20b


In [39]:
MODEL = "openai/gpt-oss-20b"

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one short line."}],
)
print(r.choices[0].message.content)

Hello!


In [40]:
def answer(query, k=4, min_score=0.0):
    hits = hybrid_rerank(query, fetch=6, k=k)
    keep = [(i, s) for i, s in hits if s > min_score]
    if not keep:
        return "I don't know based on the notes.", []
    context = "\n\n".join(f"[chunk {i+1}] {chunks[i]}" for i, _ in keep)
    prompt = f"""Answer using only the context below.
Rules:
1. If the context fully or partly answers the question, give what the context says. If the question assumes something different from the context (for example a different number), say what the context actually states.
2. Only if the context has nothing relevant, say "I don't know based on the notes."
3. Mention the chunk numbers you used.

Context:
{context}

Question: {query}
Answer:"""
    r = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.choices[0].message.content, [i + 1 for i, _ in keep]

for q in ["List the 5 steps of the RAG pipeline.",
          "How is waste sorted and annotated for AI models?",
          "How are annotation quality and embeddings related?",
          "Who won the cricket match yesterday?"]:
    text, used = answer(q)
    print(f"\nQ: {q}\nChunks used: {used}\nA: {text}")


Q: List the 5 steps of the RAG pipeline.
Chunks used: [2, 5]
A: The notes state that the RAG pipeline consists of **three** main steps:

1. **Indexing** – documents are split into chunks, each chunk is embedded into a numeric vector, and the vectors are stored in a vector database.  
2. **Retrieval** – a user’s question is embedded, and the vector database is queried for the most similar chunks using cosine similarity.  
3. **Generation** – the retrieved chunks are combined with the original question into a prompt that is fed to a language model to produce the final answer.  

(Chunk 2)

Q: How is waste sorted and annotated for AI models?
Chunks used: [6, 9, 8, 7]
A: Waste is sorted for AI models by first training computer‑vision systems to recognize and classify common waste categories—such as plastic, paper, metal, glass, organic waste, and e‑waste—from images taken on conveyor belts, bins, or sorting facilities【chunk 6】. To create the training data, annotators draw bounding boxes o

In [41]:
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

stop = {"what","is","the","a","an","of","to","and","in","how","does","do","for","are","it","this","that","with","as","by","on"}
def tok(text):
    return [w for w in re.findall(r"[a-z0-9\-]+", text.lower()) if w not in stop]

bm25 = BM25Okapi([tok(c) for c in chunks])

def hybrid_search(query, k=2, rrf_k=60):
    sem_docs = vector_db.similarity_search(query, k=len(chunks))
    sem_rank = [chunks.index(d.page_content) for d in sem_docs]
    kw_scores = bm25.get_scores(tok(query))
    kw_rank = sorted(range(len(chunks)), key=lambda i: kw_scores[i], reverse=True)
    fused = {}
    for ranking in (sem_rank, kw_rank):
        for pos, i in enumerate(ranking):
            fused[i] = fused.get(i, 0) + 1 / (rrf_k + pos + 1)
    return sorted(fused, key=fused.get, reverse=True)[:k]

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def hybrid_rerank(query, fetch=5, k=2):
    cand = hybrid_search(query, k=fetch)
    scores = reranker.predict([(query, chunks[i]) for i in cand])
    return sorted(zip(cand, scores), key=lambda x: x[1], reverse=True)[:k]

print("Retrieval functions ready")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retrieval functions ready


In [42]:
!pip install langchain langchain-community langchain-text-splitters chromadb sentence-transformers rank_bm25 groq -q

In [43]:
q = "List the 5 steps of the RAG pipeline."
text, used = answer(q)
print(f"Chunks used: {used}\nA: {text}")

Chunks used: [2, 5]
A: The notes state that the RAG pipeline consists of **three** main steps:

1. **Indexing** – documents are split into chunks, each chunk is embedded into a numeric vector, and the vectors are stored in a vector database.  
2. **Retrieval** – a user’s question is embedded, and the vector database is queried for the most similar chunks using cosine similarity.  
3. **Generation** – the retrieved chunks are combined with the original question into a prompt that is fed to a language model to produce the final answer.  

(Chunk 2)


In [44]:
for q in ["How is waste sorted and annotated for AI models?",
          "How are annotation quality and embeddings related?"]:
    print("\n", q)
    for i, s in hybrid_rerank(q, fetch=6, k=6):
        print(f"  chunk {i+1}: {s:.2f}")


 How is waste sorted and annotated for AI models?
  chunk 6: 5.02
  chunk 9: 4.86
  chunk 8: 2.43
  chunk 7: 1.70
  chunk 5: -0.97
  chunk 1: -10.45

 How are annotation quality and embeddings related?
  chunk 3: 0.82
  chunk 7: -1.03
  chunk 5: -2.37
  chunk 8: -3.53
  chunk 4: -4.92
  chunk 2: -5.56


Step: Turn the pipeline into an API (FastAPI)


In [45]:
!pip install fastapi httpx -q


In [46]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/ask")
def ask(question: str):
    text, used = answer(question)
    return {"answer": text, "chunks_used": used}

api_client = TestClient(app)
resp = api_client.get("/ask", params={"question": "What is cosine similarity?"})
print(resp.status_code)
print(resp.json())

200
{'answer': 'Cosine similarity is a measure that calculates the angle between two vectors. A smaller angle indicates a higher similarity between the vectors. (Chunk\u202f3)', 'chunks_used': [3, 2]}


In [47]:
r1 = api_client.get("/ask", params={"question": "Who won the cricket match yesterday?"})
print(r1.status_code, r1.json())

r2 = api_client.get("/ask")   # question bheja hi nahi
print(r2.status_code)


200 {'answer': "I don't know based on the notes.", 'chunks_used': []}
422


## Problems I hit and how I solved them

1. **BM25 returned the wrong chunk.** "similarity?" and "similarity," were treated as different words, and common words like "what" and "is" inflated scores. Fix: I removed punctuation and stopwords in a tokenizer. The right chunk then ranked first.

2. **Re-ranker seemed to lose a correct chunk.** I found that three chunks (7, 8, 9) all discussed annotation quality, so my label was ambiguous. Lesson: a golden set should allow more than one acceptable chunk.

3. **A relevant chunk was cut by top-k.** For one waste-sorting question, the re-ranker gave the chunk a positive score but it ranked 4th, and k=3 dropped it. Fix: k=4.

4. **Over-refusal on a false-premise question.** The model said "I don't know" to "List the 5 steps of RAG" even though the notes list 3. Fix: I split the prompt rules into "partial answer" and "nothing relevant", and the model corrected the premise.

5. **`model_not_found` from the LLM API.** The model name I used was not available. Fix: I listed the models on my account and kept the model name in one `MODEL` variable.

6. **`NameError` and `FileNotFoundError` after restart.** Variables, functions and the Drive connection are lost when the session resets. Fix: mount Drive first, run cells top to bottom, and install all libraries in the first cell.


7. **API key safety.** I stored the key in Colab Secrets, not in the code.